# Post 5: Evaluasi Model dan Praktik Machine Learning

Model machine learning yang sudah dilatih belum tentu siap digunakan. Kita perlu cara yang objektif untuk mengukur seberapa baik model bekerja pada data baru, bukan hanya data yang dipakai saat pelatihan.

Di sinilah **evaluasi model** menjadi sangat penting. Pada post ini, kita akan membahas berbagai metrik dan teknik evaluasi, mulai dari klasifikasi hingga regresi, serta cara menghadapi situasi-situasi khusus seperti data yang tidak seimbang.

---

**Yang akan dipelajari:**
- Confusion matrix, akurasi, precision, recall, dan F1-Score
- Metrik evaluasi untuk regresi (MAE, MSE, RMSE, R²)
- Masalah imbalanced dataset dan cara mengatasinya
- ROC Curve dan AUC
- Feature importance
- Studi kasus terpadu menggunakan Scikit-learn

## Daftar Isi

1. [Evaluasi Model Klasifikasi](#bagian-1)
2. [Ilustrasi Python: Train-Test Split dan Confusion Matrix](#bagian-2)
3. [Metrik Evaluasi untuk Regresi](#bagian-3)
4. [Imbalanced Dataset Problem](#bagian-4)
5. [ROC Curve dan AUC](#bagian-5)
6. [Feature Importance](#bagian-6)
7. [Studi Kasus Terpadu](#bagian-7)

## Setup

Jalankan sel berikut untuk mengimpor seluruh library yang diperlukan.

In [ ]:
# Impor semua library yang diperlukan
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    roc_curve,
    auc,
)

# Pengaturan tampilan plot
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid')

print("Semua library berhasil diimpor.")

---
<a id='bagian-1'></a>
## Bagian 1: Evaluasi Model Klasifikasi

Saat kita melatih model klasifikasi, kita perlu cara untuk mengukur seberapa baik model tersebut dalam memprediksi kelas yang benar. Ada beberapa metrik yang umum digunakan, dan masing-masing memiliki kelebihan dan kelemahan tersendiri.

Untuk memahami semua metrik ini, kita perlu memahami terlebih dahulu **confusion matrix**.

### 1.1 Confusion Matrix

Confusion matrix adalah tabel yang merangkum hasil prediksi model dibandingkan dengan nilai aktual. Bayangkan kita memiliki model yang memprediksi apakah seorang pasien menderita kanker atau tidak.

|  | **Prediksi: Negatif** | **Prediksi: Positif** |
|---|---|---|
| **Aktual: Negatif** | True Negative (TN) | False Positive (FP) |
| **Aktual: Positif** | False Negative (FN) | True Positive (TP) |

Penjelasan masing-masing kuadran:

- **True Positive (TP)**: Model memprediksi positif, dan memang benar positif.
- **True Negative (TN)**: Model memprediksi negatif, dan memang benar negatif.
- **False Positive (FP)**: Model memprediksi positif, padahal sebenarnya negatif. Disebut juga *Type I Error*.
- **False Negative (FN)**: Model memprediksi negatif, padahal sebenarnya positif. Disebut juga *Type II Error*.

Dari keempat nilai ini, kita bisa menghitung berbagai metrik evaluasi.

### 1.2 Akurasi (Accuracy)

**Akurasi** mengukur proporsi prediksi yang benar dari keseluruhan prediksi.

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

Contoh: jika dari 100 prediksi, 90 di antaranya benar, maka akurasi = 90%.

- **Kelebihan**: Mudah dipahami dan diinterpretasikan.
- **Kelemahan**: Bisa menyesatkan jika data tidak seimbang (akan dibahas di Bagian 4).

---

### 1.3 Precision

**Precision** mengukur seberapa tepat model saat memprediksi kelas positif. Dari semua yang diprediksi positif, berapa yang benar-benar positif?

$$\text{Precision} = \frac{TP}{TP + FP}$$

**Kapan penting**: saat kita ingin meminimalkan False Positive. Contoh: filter spam. Kita tidak ingin email penting masuk ke folder spam (FP).

---

### 1.4 Recall (Sensitivity)

**Recall** mengukur seberapa banyak kasus positif yang berhasil ditemukan model. Dari semua yang sebenarnya positif, berapa yang berhasil diprediksi dengan benar?

$$\text{Recall} = \frac{TP}{TP + FN}$$

**Kapan penting**: saat kita ingin meminimalkan False Negative. Contoh: deteksi penyakit. Kita tidak ingin pasien yang sakit dinyatakan sehat (FN).

---

### 1.5 F1-Score

**F1-Score** adalah rata-rata harmonik dari Precision dan Recall. Metrik ini berguna ketika kita ingin menyeimbangkan keduanya.

$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Kapan penting**: saat data tidak seimbang dan kita butuh satu angka yang merangkum performa model secara keseluruhan.

---

### Ringkasan: Kapan Menggunakan Metrik Apa?

| Situasi | Metrik yang Diutamakan |
|---|---|
| Data seimbang, interpretasi mudah | Akurasi |
| Meminimalkan False Positive | Precision |
| Meminimalkan False Negative | Recall |
| Keseimbangan antara Precision dan Recall | F1-Score |

---
<a id='bagian-2'></a>
## Bagian 2: Ilustrasi Python - Train-Test Split dan Confusion Matrix

Sekarang kita akan mempraktikkan konsep di atas menggunakan dataset bawaan Scikit-learn, yaitu **Breast Cancer Wisconsin Dataset**. Dataset ini berisi data klinis untuk mengklasifikasikan tumor menjadi ganas (*malignant*) atau jinak (*benign*).

> **Catatan tentang train-test split**: kita tidak boleh mengevaluasi model menggunakan data yang sama dengan data latihnya. Ibarat ujian, soal ujian harus berbeda dari soal latihan. Oleh karena itu, data dibagi menjadi **data latih** (untuk melatih model) dan **data uji** (untuk mengukur performa model pada data yang belum pernah dilihat).

In [ ]:
# Muat dataset breast cancer
data = load_breast_cancer()

# Konversi ke DataFrame agar lebih mudah dilihat
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = ganas, 1 = jinak

print(f"Jumlah baris   : {df.shape[0]}")
print(f"Jumlah kolom   : {df.shape[1]}")
print(f"\nDistribusi kelas:")
print(df['target'].value_counts().rename({0: 'Ganas (0)', 1: 'Jinak (1)'}))
print(f"\n5 baris pertama:")
df.head()

In [ ]:
# Pisahkan fitur (X) dan label (y)
X = data.data
y = data.target

# Bagi data menjadi data latih dan data uji
# test_size=0.2 artinya 20% untuk uji, 80% untuk latih
# random_state memastikan hasil yang sama setiap kali dijalankan
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Jumlah data latih : {X_latih.shape[0]} sampel")
print(f"Jumlah data uji   : {X_uji.shape[0]} sampel")

In [ ]:
# Standardisasi fitur agar skala data seragam
# fit_transform hanya pada data latih, transform saja pada data uji
# (menghindari kebocoran informasi dari data uji)
skaler = StandardScaler()
X_latih_skalir = skaler.fit_transform(X_latih)
X_uji_skalir   = skaler.transform(X_uji)

# Latih model regresi logistik
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_latih_skalir, y_latih)

# Prediksi pada data uji
y_prediksi = model.predict(X_uji_skalir)

print("Model selesai dilatih.")
print(f"Contoh prediksi (10 pertama): {y_prediksi[:10]}")
print(f"Label aktual   (10 pertama): {y_uji[:10]}")

In [ ]:
# Hitung semua metrik evaluasi
akurasi = accuracy_score(y_uji, y_prediksi)
presisi = precision_score(y_uji, y_prediksi)
rekol   = recall_score(y_uji, y_prediksi)
f1      = f1_score(y_uji, y_prediksi)

print("Hasil Evaluasi Model (Logistic Regression):")
print(f"  Akurasi   : {akurasi:.4f}  ({akurasi*100:.2f}%)")
print(f"  Precision : {presisi:.4f}")
print(f"  Recall    : {rekol:.4f}")
print(f"  F1-Score  : {f1:.4f}")

In [ ]:
# Buat dan tampilkan confusion matrix
cm = confusion_matrix(y_uji, y_prediksi)

tampilan_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Ganas (0)', 'Jinak (1)']
)

fig, ax = plt.subplots(figsize=(6, 5))
tampilan_cm.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix - Breast Cancer Dataset', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

print(f"True Positive (TP)  : {cm[1, 1]:3d}  (prediksi jinak, aktual jinak)")
print(f"True Negative (TN)  : {cm[0, 0]:3d}  (prediksi ganas, aktual ganas)")
print(f"False Positive (FP) : {cm[0, 1]:3d}  (prediksi jinak, aktual ganas)")
print(f"False Negative (FN) : {cm[1, 0]:3d}  (prediksi ganas, aktual jinak)")

In [ ]:
# Classification report memberikan ringkasan lengkap per kelas
print("Laporan Klasifikasi Lengkap:")
print("-" * 55)
print(classification_report(
    y_uji, y_prediksi,
    target_names=['Ganas (0)', 'Jinak (1)']
))

---
<a id='bagian-3'></a>
## Bagian 3: Metrik Evaluasi untuk Regresi

Metrik yang telah kita pelajari di atas digunakan untuk masalah **klasifikasi**, yaitu saat model memprediksi kategori. Untuk masalah **regresi** (memprediksi nilai numerik seperti harga rumah atau kadar gula darah), kita menggunakan metrik yang berbeda.

### 3.1 Metrik-Metrik Regresi

---

**Mean Absolute Error (MAE)**

MAE mengukur rata-rata selisih absolut antara nilai prediksi dan nilai aktual.

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Mudah diinterpretasikan: "rata-rata kesalahan prediksi saya adalah X satuan."

---

**Mean Squared Error (MSE)**

MSE mengkuadratkan selisih sebelum dirata-rata, sehingga error yang besar mendapat penalti yang jauh lebih tinggi.

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

---

**Root Mean Squared Error (RMSE)**

RMSE adalah akar kuadrat dari MSE. Satuannya sama dengan variabel target sehingga lebih mudah diinterpretasikan dibandingkan MSE.

$$\text{RMSE} = \sqrt{\text{MSE}}$$

---

**R-squared (R²)**

R² mengukur seberapa besar proporsi variasi data yang bisa dijelaskan oleh model. Nilainya berkisar antara 0 dan 1 (atau bisa negatif jika model sangat buruk).

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

- R² = 1.0 berarti model sempurna.
- R² = 0.0 berarti model tidak lebih baik dari sekadar menggunakan rata-rata.
- R² negatif berarti model lebih buruk dari prediksi rata-rata.

---

### 3.2 Kapan Menggunakan Metrik Apa?

| Metrik | Kelebihan | Gunakan Ketika |
|---|---|---|
| MAE | Mudah dipahami, tidak terlalu sensitif terhadap outlier | Outlier tidak terlalu penting |
| MSE | Menghukum error besar lebih keras | Ingin meminimalkan error besar |
| RMSE | Satuan sama dengan target, mudah diinterpretasikan | Ingin nilai error dalam satuan aslinya |
| R² | Menunjukkan kualitas model secara keseluruhan | Ingin tahu seberapa baik model menjelaskan variasi data |

In [ ]:
# Gunakan dataset diabetes bawaan Scikit-learn
# Target: ukuran perkembangan penyakit satu tahun ke depan (nilai numerik)
data_regresi = load_diabetes()

X_reg = data_regresi.data
y_reg = data_regresi.target

# Bagi data latih dan uji
X_latih_r, X_uji_r, y_latih_r, y_uji_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Latih model regresi linear
model_reg = LinearRegression()
model_reg.fit(X_latih_r, y_latih_r)

# Prediksi
y_pred_r = model_reg.predict(X_uji_r)

# Hitung metrik
mae  = mean_absolute_error(y_uji_r, y_pred_r)
mse  = mean_squared_error(y_uji_r, y_pred_r)
rmse = np.sqrt(mse)
r2   = r2_score(y_uji_r, y_pred_r)

print("Metrik Evaluasi Regresi - Dataset Diabetes:")
print(f"  MAE  : {mae:.2f}")
print(f"  MSE  : {mse:.2f}")
print(f"  RMSE : {rmse:.2f}")
print(f"  R²   : {r2:.4f}")

In [ ]:
# Visualisasi nilai aktual vs nilai prediksi
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(y_uji_r, y_pred_r, alpha=0.6, color='steelblue', label='Data Uji')
ax.plot(
    [y_uji_r.min(), y_uji_r.max()],
    [y_uji_r.min(), y_uji_r.max()],
    'r--', lw=2, label='Prediksi Sempurna'
)

ax.set_xlabel('Nilai Aktual')
ax.set_ylabel('Nilai Prediksi')
ax.set_title(f'Aktual vs Prediksi - Dataset Diabetes\n(R² = {r2:.4f})', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

---
<a id='bagian-4'></a>
## Bagian 4: Imbalanced Dataset Problem

### 4.1 Apa Itu Class Imbalance?

Bayangkan kita memiliki dataset deteksi penipuan kartu kredit dengan 990 transaksi normal dan 10 transaksi penipuan. Jika model kita selalu memprediksi "normal" untuk semua transaksi, akurasi yang didapat adalah 99%. Kedengarannya bagus, namun model tersebut sama sekali tidak berguna karena tidak berhasil mendeteksi satu pun penipuan.

Inilah yang disebut **masalah imbalanced dataset** atau ketidakseimbangan kelas.

### 4.2 Mengapa Akurasi Bisa Menyesatkan

Saat distribusi kelas sangat tidak seimbang, model bisa "curang" dengan selalu memprediksi kelas mayoritas. Akurasi tetap tinggi, tapi model tidak belajar apa pun tentang kelas minoritas yang justru sering menjadi fokus perhatian.

In [ ]:
# Simulasi dataset tidak seimbang: 990 normal, 10 penipuan
np.random.seed(42)

y_aktual = np.array([0] * 990 + [1] * 10)
y_naive  = np.zeros(1000, dtype=int)  # model selalu prediksi 0 (normal)

akurasi_naive = accuracy_score(y_aktual, y_naive)
presisi_naive = precision_score(y_aktual, y_naive, zero_division=0)
rekol_naive   = recall_score(y_aktual, y_naive, zero_division=0)
f1_naive      = f1_score(y_aktual, y_naive, zero_division=0)

print("Model Naif (selalu prediksi Normal):")
print(f"  Akurasi   : {akurasi_naive:.2%}  <-- terlihat sangat baik!")
print(f"  Precision : {presisi_naive:.2%}")
print(f"  Recall    : {rekol_naive:.2%}  <-- tidak mendeteksi satupun penipuan")
print(f"  F1-Score  : {f1_naive:.2%}")
print()
print("Kesimpulan: akurasi tinggi, tapi model tidak berguna sama sekali.")

In [ ]:
# Visualisasi distribusi kelas yang tidak seimbang
fig, ax = plt.subplots(figsize=(5, 4))

kelas  = ['Normal (0)', 'Penipuan (1)']
jumlah = [990, 10]
warna  = ['steelblue', 'tomato']

batang = ax.bar(kelas, jumlah, color=warna, edgecolor='white', linewidth=1.5)

for b in batang:
    tinggi = b.get_height()
    ax.text(
        b.get_x() + b.get_width() / 2.,
        tinggi + 15,
        f'{tinggi} ({tinggi / sum(jumlah) * 100:.1f}%)',
        ha='center', va='bottom', fontsize=11
    )

ax.set_ylabel('Jumlah Sampel')
ax.set_title('Distribusi Kelas yang Tidak Seimbang', fontsize=13)
ax.set_ylim(0, 1150)
plt.tight_layout()
plt.show()

### 4.3 Teknik Mengatasi Imbalanced Dataset

Terdapat beberapa pendekatan umum untuk menangani data tidak seimbang:

---

**1. Oversampling - memperbanyak kelas minoritas**

Menambahkan sampel dari kelas yang lebih sedikit agar jumlahnya setara dengan kelas mayoritas.

Teknik populer: **SMOTE (Synthetic Minority Oversampling Technique)**. SMOTE tidak sekadar menduplikasi data yang ada, melainkan membuat sampel sintetis baru berdasarkan interpolasi antara sampel yang sudah ada.

```python
# Contoh penggunaan SMOTE (memerlukan library imbalanced-learn)
# pip install imbalanced-learn

from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_seimbang, y_seimbang = sm.fit_resample(X_latih, y_latih)
```

---

**2. Undersampling - mengurangi kelas mayoritas**

Mengurangi sampel dari kelas yang lebih banyak agar setara dengan kelas minoritas. Sederhana, namun berisiko membuang informasi penting.

---

**3. Penyesuaian bobot kelas (Class Weight)**

Memberikan penalti lebih besar pada model ketika salah memprediksi kelas minoritas. Banyak algoritma Scikit-learn mendukung parameter `class_weight='balanced'`.

```python
# Contoh penggunaan class_weight pada Logistic Regression
model_seimbang = LogisticRegression(class_weight='balanced', random_state=42)
```

---

**Rekomendasi umum**: saat menghadapi imbalanced dataset, gunakan F1-Score, Precision, dan Recall sebagai metrik utama (bukan akurasi semata).

---
<a id='bagian-5'></a>
## Bagian 5: ROC Curve dan AUC

### 5.1 Konsep Dasar: True Positive Rate vs False Positive Rate

Model klasifikasi biasanya menghasilkan probabilitas, bukan langsung label kelas. Ambang batas (*threshold*) default yang digunakan adalah 0.5: jika probabilitas lebih dari 0.5, model memprediksi kelas positif.

Namun, threshold ini bisa diubah-ubah. **ROC Curve** (Receiver Operating Characteristic Curve) memvisualisasikan performa model pada berbagai nilai threshold sekaligus.

ROC Curve memplot dua nilai:

- **True Positive Rate (TPR) / Recall**: proporsi kasus positif yang berhasil terdeteksi.
$$TPR = \frac{TP}{TP + FN}$$

- **False Positive Rate (FPR)**: proporsi kasus negatif yang keliru diprediksi positif.
$$FPR = \frac{FP}{FP + TN}$$

Sumbu X adalah FPR, dan sumbu Y adalah TPR. Setiap titik pada kurva mewakili satu nilai threshold.

### 5.2 Interpretasi AUC Score

**AUC (Area Under the Curve)** adalah luas area di bawah ROC Curve, yang merangkum performa model menjadi satu angka tunggal.

| Nilai AUC | Interpretasi |
|---|---|
| 1.0 | Model sempurna |
| 0.9 - 1.0 | Sangat baik |
| 0.8 - 0.9 | Baik |
| 0.7 - 0.8 | Cukup |
| 0.5 - 0.7 | Kurang baik |
| 0.5 | Tidak lebih baik dari tebak acak |
| < 0.5 | Lebih buruk dari tebak acak |

In [ ]:
# Ambil probabilitas prediksi untuk kelas positif (jinak = 1)
# predict_proba mengembalikan probabilitas untuk setiap kelas
y_prob = model.predict_proba(X_uji_skalir)[:, 1]

# Hitung TPR dan FPR untuk berbagai nilai threshold
fpr, tpr, threshold_roc = roc_curve(y_uji, y_prob)
nilai_auc = auc(fpr, tpr)

print(f"Nilai AUC: {nilai_auc:.4f}")

In [ ]:
# Plot ROC Curve
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'ROC Curve (AUC = {nilai_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--',
        label='Tebak Acak (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.1, color='darkorange')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('ROC Curve - Breast Cancer Dataset', fontsize=13)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
<a id='bagian-6'></a>
## Bagian 6: Feature Importance

### 6.1 Cara Melihat Fitur yang Paling Berpengaruh

Tidak semua fitur dalam dataset berkontribusi secara setara terhadap prediksi model. **Feature importance** membantu kita memahami fitur mana yang paling berpengaruh dalam pengambilan keputusan model.

Memahami feature importance berguna untuk:

- **Interpretasi model**: memahami faktor apa yang paling menentukan prediksi.
- **Seleksi fitur**: menghapus fitur yang tidak relevan untuk menyederhanakan model.
- **Domain insight**: mendapatkan pemahaman baru tentang data dari sudut pandang model.

### 6.2 Interpretability Model Sederhana

Tidak semua model memberikan feature importance secara langsung. Algoritma berbasis pohon keputusan seperti **Random Forest** dan **Gradient Boosting** menghasilkan feature importance berdasarkan seberapa sering dan seberapa efektif sebuah fitur digunakan untuk membagi data di setiap pohon.

In [ ]:
# Latih model Random Forest pada dataset breast cancer
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_latih_skalir, y_latih)

# Ambil nilai feature importance
importansi = rf_model.feature_importances_
nama_fitur = data.feature_names

# Buat DataFrame dan urutkan dari yang terpenting
df_importansi = pd.DataFrame({
    'Fitur': nama_fitur,
    'Importansi': importansi
}).sort_values('Importansi', ascending=False)

print("10 Fitur Terpenting (Random Forest):")
print("-" * 45)
print(df_importansi.head(10).to_string(index=False))

In [ ]:
# Visualisasi 10 fitur terpenting
top10 = df_importansi.head(10)

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    top10['Fitur'][::-1],
    top10['Importansi'][::-1],
    color='steelblue',
    edgecolor='white',
    linewidth=0.8
)

ax.set_xlabel('Nilai Importansi')
ax.set_title('10 Fitur Terpenting - Random Forest\n(Breast Cancer Dataset)', fontsize=13)
plt.tight_layout()
plt.show()

---
<a id='bagian-7'></a>
## Bagian 7: Studi Kasus Terpadu

Pada bagian ini, kita akan menggabungkan seluruh konsep yang telah dipelajari ke dalam satu alur kerja yang lengkap menggunakan **Breast Cancer Wisconsin Dataset**.

Alur kerja:
1. Muat dan eksplorasi data
2. Bagi data dengan train-test split
3. Latih dua model berbeda (Logistic Regression dan Random Forest)
4. Evaluasi dan bandingkan performa keduanya
5. Visualisasikan confusion matrix dan ROC Curve secara berdampingan
6. Analisis feature importance

In [ ]:
# LANGKAH 1: Muat data
data_kasus = load_breast_cancer()
X_kasus = data_kasus.data
y_kasus = data_kasus.target

print("Informasi Dataset:")
print(f"  Jumlah sampel : {X_kasus.shape[0]}")
print(f"  Jumlah fitur  : {X_kasus.shape[1]}")
print(f"  Kelas         : {data_kasus.target_names.tolist()}")
print()

unik, jumlah = np.unique(y_kasus, return_counts=True)
print("Distribusi kelas:")
for u, j in zip(unik, jumlah):
    print(f"  Kelas {u} ({data_kasus.target_names[u]:10s}): {j} sampel ({j/len(y_kasus)*100:.1f}%)")

In [ ]:
# LANGKAH 2: Bagi data dan standardisasi
X_latih_k, X_uji_k, y_latih_k, y_uji_k = train_test_split(
    X_kasus, y_kasus, test_size=0.2, random_state=42, stratify=y_kasus
)

skaler_k    = StandardScaler()
X_latih_k_s = skaler_k.fit_transform(X_latih_k)
X_uji_k_s   = skaler_k.transform(X_uji_k)

print(f"Data latih : {X_latih_k_s.shape[0]} sampel")
print(f"Data uji   : {X_uji_k_s.shape[0]} sampel")

In [ ]:
# LANGKAH 3: Latih dua model
model_lr = LogisticRegression(random_state=42, max_iter=1000)
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)

model_lr.fit(X_latih_k_s, y_latih_k)
model_rf.fit(X_latih_k_s, y_latih_k)

pred_lr = model_lr.predict(X_uji_k_s)
pred_rf = model_rf.predict(X_uji_k_s)

print("Kedua model selesai dilatih.")

In [ ]:
# LANGKAH 4: Bandingkan performa
def hitung_metrik(nama, y_aktual, y_pred):
    return {
        'Model'    : nama,
        'Akurasi'  : accuracy_score(y_aktual, y_pred),
        'Precision': precision_score(y_aktual, y_pred),
        'Recall'   : recall_score(y_aktual, y_pred),
        'F1-Score' : f1_score(y_aktual, y_pred),
    }

hasil = pd.DataFrame([
    hitung_metrik('Logistic Regression', y_uji_k, pred_lr),
    hitung_metrik('Random Forest',       y_uji_k, pred_rf),
])

# Format angka
hasil_fmt = hasil.copy()
for kol in ['Akurasi', 'Precision', 'Recall', 'F1-Score']:
    hasil_fmt[kol] = hasil_fmt[kol].apply(lambda x: f"{x:.4f}")

print("Perbandingan Performa Model:")
print("-" * 60)
print(hasil_fmt.to_string(index=False))

In [ ]:
# LANGKAH 5a: Confusion matrix berdampingan
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, judul in zip(
    [ax1, ax2],
    [pred_lr, pred_rf],
    ['Logistic Regression', 'Random Forest']
):
    cm = confusion_matrix(y_uji_k, pred)
    tampilan = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=data_kasus.target_names
    )
    tampilan.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(judul, fontsize=12)

plt.suptitle('Confusion Matrix - Perbandingan Model', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# LANGKAH 5b: ROC Curve berdampingan
prob_lr = model_lr.predict_proba(X_uji_k_s)[:, 1]
prob_rf = model_rf.predict_proba(X_uji_k_s)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_uji_k, prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_uji_k, prob_rf)

auc_lr = auc(fpr_lr, tpr_lr)
auc_rf = auc(fpr_rf, tpr_rf)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(fpr_lr, tpr_lr, lw=2, label=f'Logistic Regression (AUC = {auc_lr:.4f})')
ax.plot(fpr_rf, tpr_rf, lw=2, label=f'Random Forest       (AUC = {auc_rf:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Tebak Acak')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('Perbandingan ROC Curve - Studi Kasus Terpadu', fontsize=13)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# LANGKAH 6: Feature importance dari Random Forest
importansi_k = model_rf.feature_importances_
df_imp_k = pd.DataFrame({
    'Fitur'     : data_kasus.feature_names,
    'Importansi': importansi_k
}).sort_values('Importansi', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    df_imp_k['Fitur'][::-1],
    df_imp_k['Importansi'][::-1],
    color='steelblue',
    edgecolor='white',
    linewidth=0.8
)
ax.set_xlabel('Nilai Importansi')
ax.set_title('10 Fitur Terpenting - Random Forest\n(Studi Kasus Terpadu)', fontsize=13)
plt.tight_layout()
plt.show()

---

## Penutup

Selamat, kamu telah menyelesaikan seluruh pembahasan tentang evaluasi model machine learning. Berikut adalah ringkasan dari semua yang telah dipelajari:

| Topik | Poin Utama |
|---|---|
| Confusion Matrix | Dasar dari semua metrik klasifikasi: TP, TN, FP, FN |
| Akurasi | Mudah diinterpretasikan, tapi bisa menyesatkan pada data tidak seimbang |
| Precision dan Recall | Saling trade-off; pilih sesuai konteks dan risiko yang ingin dihindari |
| F1-Score | Keseimbangan antara Precision dan Recall |
| Metrik Regresi | MAE, MSE, RMSE, dan R² masing-masing punya keunggulan yang berbeda |
| Imbalanced Dataset | Akurasi saja tidak cukup; gunakan F1, Recall, atau teknik resampling |
| ROC Curve dan AUC | Evaluasi performa lintas threshold; AUC mendekati 1.0 berarti lebih baik |
| Feature Importance | Membantu interpretasi model dan seleksi fitur |

**Langkah selanjutnya yang bisa dijelajahi:**
- Cross-validation untuk evaluasi yang lebih robust dan tidak bergantung pada satu pembagian data
- Hyperparameter tuning dengan `GridSearchCV` atau `RandomizedSearchCV`
- Pipeline Scikit-learn untuk menggabungkan preprocessing dan model dalam satu objek
- Model deployment: dari notebook ke aplikasi nyata